In [1]:
##  一个售前和售后的 langchain  LLMRouterChain 模版

from langchain_classic.chains.router import MultiPromptChain
from langchain_community.llms import Tongyi
from langchain_classic.chains import ConversationChain
from langchain_classic.chains.llm import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.router.llm_router import (
    LLMRouterChain,
    RouterOutputParser
)
from langchain_classic.chains.router.multi_prompt_prompt import (
    MULTI_PROMPT_ROUTER_TEMPLATE
)

# 售前咨询模板
presales_prompt_tpl = PromptTemplate.from_template(
    '你是一位专业的售前顾问，擅长产品介绍、方案推荐和商务咨询。'
    '你需要热情、专业地回答客户的产品咨询、价格询问、功能介绍等售前问题。'
    '请使用中文帮我解答下列售前咨询问题：\n{input}'
)

# 售后服务模板
aftersales_prompt_tpl = PromptTemplate.from_template(
    '你是一位耐心的售后服务专员，擅长解决客户的使用问题、技术支持和投诉处理。'
    '你需要耐心、细致地帮助客户解决产品使用中遇到的问题，提供技术支持和服务指导。'
    '请使用中文帮我解答下列售后服务问题：\n{input}'
)

# 创建模板信息列表
prompt_infos = [
    {
        'name': 'presales',
        'description': '用于处理售前咨询，包括产品介绍、价格询问、功能说明、方案推荐等',
        'prompt_template': presales_prompt_tpl,
    },
    {
        'name': 'aftersales',
        'description': '用于处理售后服务，包括使用问题、技术支持、故障排除、投诉处理等',
        'prompt_template': aftersales_prompt_tpl,
    },
]

llm = Tongyi(
    temperature=0.1,
)

# 生成键为模板名称、值为Chain的字典
destination_chains = {}
for p_info in prompt_infos:
    name = p_info['name']
    prompt = p_info['prompt_template']
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain

# 将模板名称和模板描述通过MULTI_PROMPT_ROUTER_TEMPLATE生成模板
destinations = [f'{p["name"]}: {p["description"]}'
                for p in prompt_infos]
destinations_str = "\n".join(destinations)

router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=['input'],
    output_parser=RouterOutputParser(),
)
router_chain = LLMRouterChain.from_llm(llm, router_prompt)

# 这里创建了一个default_chain
# 为了防止提的问题类型并没有包含在prompt_infos中
default_chain = ConversationChain(llm=llm, output_key='text')
chain = MultiPromptChain(
    router_chain=router_chain,
    destination_chains=destination_chains,
    default_chain=default_chain,
    verbose=True,
)

# 测试售前咨询问题
print("=== 售前咨询测试 ===")
print(chain.run("你们的产品有什么功能？价格是多少？"))

print("\n=== 售后服务测试 ===")
print(chain.run("我的产品出现故障了，无法正常启动，该怎么办？"))

print("\n=== 其他问题测试 ===")
print(chain.run("今天天气怎么样？"))


/Users/anthony/miniforge3/envs/new_ai_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/77/wn2nxhbs4rx6vwtr4mkcww780000gn/T/ipykernel_40194/4193506374.py:53: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt)
/var/folders/77/wn2nxhbs4rx6vwtr4mkcww780000gn/T/ipykernel_40194/4193506374.py:73: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use `langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  default_chain = ConversationChain(llm=llm, output_key='text')
/Users/anthony/miniforge3/envs/new_ai_env/lib/python3.11/site-packages/pydantic/main.py:

=== 售前咨询测试 ===


> Entering new MultiPromptChain chain...
presales: {'input': '你们的产品有哪些主要功能？价格是多少？'}
> Finished chain.
您好！非常感谢您对我们产品的关注，我很乐意为您详细介绍。

我们的产品是一套集智能化、高效化和可扩展性于一体的企业级解决方案，主要面向数字化转型中的各类组织，涵盖以下几大核心功能：

### 一、主要功能

1. **统一平台管理**  
   提供集中化的控制台，支持多终端、多角色的权限分配与系统监控，帮助企业实现资源的统一调度与可视化管理。

2. **智能数据分析与报表**  
   内置强大的数据引擎，支持实时数据采集、清洗、分析，并自动生成可视化图表与定制化报表，助力管理层快速决策。

3. **自动化流程引擎**  
   支持业务流程的自定义配置（如审批流、工单流转等），大幅提升运营效率，减少人工干预。

4. **开放API接口与系统集成**  
   提供标准化API接口，可轻松对接ERP、CRM、OA等第三方系统，实现数据互通与业务协同。

5. **安全与合规保障**  
   支持数据加密传输、访问审计、等保合规支持，确保企业信息资产安全可靠。

6. **移动端支持**  
   提供iOS/Android App及H5页面，支持随时随地处理任务与查看数据。

---

### 二、价格说明

我们的产品采用**灵活的订阅制计费模式**，具体价格根据以下因素综合确定：

- **用户规模**：按账号数量阶梯计价（例如：50人以内、50–200人、200人以上）  
- **功能模块选择**：基础版、标准版、专业版可选，不同版本功能丰富度不同  
- **部署方式**：支持公有云（SaaS）、私有云或本地化部署，部署方式影响成本  
- **服务周期**：年付享优惠，通常比月付节省15%~20%

🔹 **参考报价范围**（以SaaS标准版为例）：
- 基础版：¥8,000/年（适合50人以内团队，含核心功能）
- 标准版：¥18,000/年（推荐多数企业使用，含自动化+数据分析）
- 专业版：¥35,000/年起（支持高级集成、定制开发与专属支持）

> 注：以上为初步参考价，实际报价将结合您的具体业务需求

In [2]:
# 整合链的语法

from langchain_classic.chains.router import MultiPromptChain
from langchain_community.llms import Tongyi
from langchain_classic.chains import ConversationChain
from langchain_classic.chains.llm import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.router.llm_router import (
    LLMRouterChain,
    RouterOutputParser
)
from langchain_classic.chains.router.multi_prompt_prompt import (
    MULTI_PROMPT_ROUTER_TEMPLATE
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# 初始化LLM
llm = Tongyi(temperature=0.1)

# 售前咨询链 - 使用新式语法
presales_prompt = PromptTemplate.from_template(
    '你是一位专业的售前顾问，擅长产品介绍、方案推荐和商务咨询。'
    '你需要热情、专业地回答客户的产品咨询、价格询问、功能介绍等售前问题。'
    '请使用中文帮我解答下列售前咨询问题：\n{input}'
)
presales_chain = presales_prompt | llm | StrOutputParser()

# 售后服务链 - 使用新式语法
aftersales_prompt = PromptTemplate.from_template(
    '你是一位耐心的售后服务专员，擅长解决客户的使用问题、技术支持和投诉处理。'
    '你需要耐心、细致地帮助客户解决产品使用中遇到的问题，提供技术支持和服务指导。'
    '请使用中文帮我解答下列售后服务问题：\n{input}'
)
aftersales_chain = aftersales_prompt | llm | StrOutputParser()

# 意图识别链 - 使用新式语法
intent_prompt = PromptTemplate.from_template(
    """请分析以下用户问题的意图，判断是售前咨询还是售后服务：

售前咨询：产品介绍、功能说明、价格询问、方案推荐、购买咨询等
售后服务：使用问题、技术支持、故障排除、投诉处理、维修服务等

用户问题：{input}

请只回答"售前"或"售后"，不要添加其他内容。"""
)
intent_chain = intent_prompt | llm | StrOutputParser()

# 创建路由函数
def route_question(input_dict):
    question = input_dict["input"]
    intent = intent_chain.invoke({"input": question})
    
    print(f"识别意图: {intent.strip()}")
    
    if "售前" in intent:
        return presales_chain.invoke({"input": question})
    elif "售后" in intent:
        return aftersales_chain.invoke({"input": question})
    else:
        # 默认处理
        default_prompt = PromptTemplate.from_template(
            "我是一个智能助手，很高兴为您服务。请问有什么可以帮助您的吗？\n问题：{input}"
        )
        default_chain = default_prompt | llm | StrOutputParser()
        return default_chain.invoke({"input": question})

# 创建完整的路由链
router_chain = RunnablePassthrough() | RunnableLambda(route_question)

# 方法二：更简洁的条件路由实现
from langchain_core.runnables import RunnableBranch

# 创建条件判断函数
def is_presales(input_dict):
    intent = intent_chain.invoke(input_dict)
    return "售前" in intent

def is_aftersales(input_dict):
    intent = intent_chain.invoke(input_dict)
    return "售后" in intent

# 使用 RunnableBranch 创建条件路由
branch_chain = RunnableBranch(
    (is_presales, presales_chain),
    (is_aftersales, aftersales_chain),
    # 默认链
    PromptTemplate.from_template("我是智能助手，请问有什么可以帮助您的？\n{input}") | llm | StrOutputParser()
)

# 测试代码
if __name__ == "__main__":
    print("=== 方法一：自定义路由函数 ===")
    
    # 测试售前问题
    print("\n--- 售前咨询测试 ---")
    result1 = router_chain.invoke({"input": "你们的产品有什么功能？价格是多少？"})
    print(f"回答: {result1}")
    
    # 测试售后问题
    print("\n--- 售后服务测试 ---")
    result2 = router_chain.invoke({"input": "我的产品出现故障了，无法正常启动，该怎么办？"})
    print(f"回答: {result2}")
    
    print("\n=== 方法二：RunnableBranch 条件路由 ===")
    
    # 测试售前问题
    print("\n--- 售前咨询测试 ---")
    result3 = branch_chain.invoke({"input": "我想了解一下你们的服务套餐和收费标准"})
    print(f"回答: {result3}")
    
    # 测试售后问题
    print("\n--- 售后服务测试 ---")
    result4 = branch_chain.invoke({"input": "产品使用过程中遇到了错误提示，需要技术支持"})
    print(f"回答: {result4}")


=== 方法一：自定义路由函数 ===

--- 售前咨询测试 ---
识别意图: 售前
回答: 您好！非常感谢您对我们产品的关注，我是您的售前顾问，很高兴为您服务！

我们提供的是面向企业级用户的智能化解决方案，具体产品功能和价格会根据您的实际业务需求进行个性化配置。以下是我们的核心产品功能概览：

🔹 主要功能包括：
1. **智能数据管理**：支持多源数据接入、清洗、整合与可视化分析，助力企业实现数据驱动决策。
2. **自动化流程引擎**：可自定义审批流、任务分配和业务流程，显著提升运营效率。
3. **AI辅助能力**：集成自然语言处理、智能推荐和预测分析，适用于客服、营销、风控等场景。
4. **高安全性与合规性**：支持等保三级、GDPR等安全标准，保障数据隐私与系统稳定。
5. **开放API接口**：便于与ERP、CRM、OA等现有系统无缝对接，快速完成数字化集成。

💰 关于价格：
我们的产品采用“按需订阅”模式，价格根据用户数、功能模块、部署方式（公有云/私有化部署）以及服务等级等因素综合确定。例如：
- 标准SaaS版：适合中小型企业，起价为 **9,800元/年**，包含基础功能与云端托管；
- 专业定制版：支持私有化部署与深度定制，通常在 **20万元起**，具体方案可进一步沟通后出具详细报价。

📌 为了给您更精准的推荐和报价，能否请您简单介绍一下：
- 贵公司的行业和主要业务？
- 您希望解决哪些具体问题或提升哪方面的能力？

我将据此为您量身定制一套高性价比的解决方案。期待您的回复！😊

--- 售后服务测试 ---
识别意图: 售后
回答: 您好，非常理解您遇到产品无法启动的困扰，感谢您及时反馈问题。为了更好地帮助您排查和解决问题，请您先不要着急，我们可以一步步来检查。以下是几个常见的排查步骤，您可以根据情况逐一尝试：

1. **检查电源连接**  
   - 请确认电源线是否插紧，插座是否有电（可尝试用其他电器测试插座）。  
   - 如果是使用电池供电的产品，请检查电池是否安装正确、电量是否充足，或尝试更换新电池。

2. **查看指示灯或声音反应**  
   - 按下电源键时，产品是否有任何反应？比如指示灯闪烁、发出声响等。  
   - 若完全无反应，可能是电源模块或内部电路问题；若有反应但无法启动，可能是系统卡住。

## EmbeddingRouterChain

不仅可以使用 LLMRouteChain 来智能选择合适的处理链，还可以采用 EmbeddingRouterChain，该组件通过计算各 Chain 描述与用户问题之间的语义相关性，实现更精准的路由决策。


In [3]:
!pip install chromadb

In [4]:
from langchain_community.vectorstores import Chroma    # # pip install chroma
from langchain_community.embeddings import DashScopeEmbeddings # pip install dashscope
from langchain_classic.chains import LLMRouterChain, MultiPromptChain
from langchain_core.language_models import BaseLLM
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import Tongyi  # 或你使用的 LLM
import os

# 1. 定义任务名称与描述
names_and_descriptions = [
    ("physics", ["用于解答物理相关问题，例如力学、电磁学等"]), 
    ("math", ["用于解答数学相关问题，例如代数、几何、微积分等"]), 
]

# 2. 使用通义千问的 Embedding 模型
embeddings = DashScopeEmbeddings(model="text-embedding-v2")

# 3. 构建向量数据库（用于路由匹配）
descriptions = []
names = []
for name, desc_list in names_and_descriptions:
    for desc in desc_list:
        descriptions.append(desc)
        names.append(name)

# 创建 Chroma 向量库
vectorstore = Chroma(embedding_function=embeddings)
# 批量添加文档
vectorstore.add_texts(texts=descriptions, metadatas=[{"name": name} for name in names])

# 4. 自定义 Embedding 路由链（LangChain 没有直接 from_names_and_descriptions）
def get_relevant_chain_name(question: str) -> str:
    docs = vectorstore.similarity_search(question, k=1)
    return docs[0].metadata["name"]

# 5. 定义各个目标链的 prompt 和 LLM
llm = Tongyi(model_name="qwen-plus", temperature=0.1)  # 可替换为你用的 LLM

physics_prompt = PromptTemplate(
    template="你是一个物理专家，请回答以下问题：\n{input}",
    input_variables=["input"]
)
math_prompt = PromptTemplate(
    template="你是一个数学专家，请回答以下问题：\n{input}",
    input_variables=["input"]
)

destination_chains = {
    "physics": physics_prompt | llm,
    "math": math_prompt | llm,
}

default_chain = PromptTemplate.from_template("请回答以下问题：{input}") | llm

# 6. 定义运行逻辑（模拟 MultiPromptChain）
def run_router_chain(question: str):
    chain_name = get_relevant_chain_name(question)
    print(f"路由到: {chain_name}")
    if chain_name in destination_chains:
        return destination_chains[chain_name].invoke({"input": question})
    else:
        return default_chain.invoke({"input": question})

# 7. 测试
result = run_router_chain("牛顿第一定律是什么？")
print(result)

/var/folders/77/wn2nxhbs4rx6vwtr4mkcww780000gn/T/ipykernel_40194/1141448552.py:27: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(embedding_function=embeddings)


路由到: physics
当然，作为一名物理专家，我很乐意为你解答。

牛顿第一定律，也称为**惯性定律**，其内容如下：

> **任何物体都会保持静止状态或者匀速直线运动状态，除非有外力迫使它改变这种状态。**

换句话说：
- 如果一个物体没有受到外力作用（或所受合外力为零），那么：
  - 原来静止的物体将继续保持静止；
  - 原来运动的物体将保持其速度（大小和方向）不变，做匀速直线运动。

这一定律揭示了物体具有保持原有运动状态的性质，这种性质被称为**惯性**。因此，牛顿第一定律也被称为**惯性定律**。

**关键点：**
- 强调了“外力”是改变物体运动状态的原因；
- 定义了惯性参考系的存在——只有在惯性参考系中，牛顿第一定律才成立；
- 是牛顿力学体系的基础之一。

举个例子：
在光滑冰面上滑行的冰球，由于摩擦力极小，几乎不受水平方向的外力，因此会近似保持匀速直线运动，直到碰到障碍物或受到其他外力作用。

希望这个解释清晰准确！如果你还想了解第二、第三定律，也可以继续提问。
